# ⚡ LightGBM: Synthetic Artifacts & Dual-Target Encoding 
**CV: 0.94587 | LB: 0.94612**

This notebook is the result of rigorous A/B testing of various feature engineering techniques discussed in the community. By isolating synthetic generator flaws and combining them with Dual-Target Encoding, this single LightGBM model achieves competitive score.

### 🙏 Acknowledgements & Credits
Special thanks to the community members whose research, notebooks, and forum posts directly shaped this approach:

**Notebooks:**
*   [**cstdy**](https://www.kaggle.com/code/kirill0212) - Inspired the baseline structure, Target Encoding strategy, and numerical digit extraction.
*   [**Evgeniy Dvorkin**](https://www.kaggle.com/code/evgendvorkin) - Inspired the robust Frequency Encoding application across all features.

**Community Discussions (The "Magic" Features):**
*   [**starkhushi & Tilii**](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/738968) - Uncovered the deterministic flaws: The "Millionaire Cliff" (>=170k) and the $30k mode collapse.
*   [**broccoli beef**](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/739142) - Proved that Logistic Regression captures the "original" dataset's true smooth curve, pushing me to anchor my model using `orig.csv` means.
*   [**Chris Deotte**](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/738991) - Highlighted the Simpson's Paradox regarding home/public charging, validating the need for deep interaction trees.

If you find this notebook helpful, please consider upvoting the linked resources above as well!

# ⚡ Pure Update: Multi-Scale Binned Numerics & Triple-TE
### 🏆 CV: 0.94607 | LB: 0.94638

> **Version 3 Update:** Integrated Markus's Multi-Scale "Smooth Keys" binning with a Triple-Target Encoding strategy (`auto`, `10.0`, `100.0`), pushing local CV from `0.94587` $\rightarrow$ `0.94607` and Public LB from `0.94612` $\rightarrow$ `0.94638`!


### 📊 Model Score History

| Version | Features & Improvements | Local 5-Fold CV | Public LB |
| :--- | :--- | :---: | :---: |
| **V1** | Dual TE (`auto`, `10`) + Digits + CTGAN Magic Flags | `0.94587` | `0.94612` |
| **V3 (Current)** | **+ Multi-Scale Income/Commute Bins + Triple TE (`100.0`)** | **`0.94607`** | **`0.94638`** |


### 💡 What's New in V3?
In V3, added engineered **Multi-Resolution "Smooth Keys"** for `Annual_Income_USD` and `Daily_Commute_km` (exact integer, `/100` floor, `/1000` floor). Then expanded the encoding engine to a **Triple Target Encoder** (`smooth='auto'`, `10.0`, `100.0`). The `100.0` heavy smoothing parameter acts as a Bayesian prior regulator, keeping micro-bins from overfitting while preserving critical CTGAN synthetic signals.


### **🙏 Acknowledgements & Credits**
This notebook update is built using:
*   [**Markus.JM notebook: S6E9 CTBoost(not catboost) Astra baseline**](https://www.kaggle.com/maiernator) - Brilliant **Multi-Scale "Smooth Keys"** income/commute binning concept and the heavy `smooth=100.0` Target Encoding parameter introduced in his CTBoost baseline.

 Please consider upvoting the original work!

In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 500)

# 1. LOAD DATA

In [2]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e9/train.csv'
TEST_PATH  = '/kaggle/input/competitions/playground-series-s6e9/test.csv'
SUB_PATH   = '/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv'
ORIG_PATH  = '/kaggle/input/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety/EV_Adoption_and_Range_Anxiety_Dataset.csv' # 

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
orig = pd.read_csv(ORIG_PATH)
submission = pd.read_csv(SUB_PATH)

train

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
668660,668660,30,71090.0,27.4,2,5,6,5.0,Male,Urban,Sedan,No,Yes,Medium,No
668661,668661,32,129990.0,5.0,2,5,4,1.0,Female,Suburban,SUV,Yes,Yes,Low,No
668662,668662,64,121791.0,24.2,1,5,6,1.0,Female,Suburban,Hatchback,Yes,Yes,Low,No
668663,668663,51,115923.0,43.7,2,0,0,1.0,Female,Rural,SUV,Yes,Yes,Low,No


# 2.  FEATURE ENGINEERING

In [3]:
TARGET = 'Will_Buy_EV'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
orig[TARGET] = orig[TARGET].map({'Yes': 1, 'No': 0})

train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined.drop(columns=['Number_of_Cars_Owned'], inplace=True, errors='ignore')

cat_cols = combined.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = [c for c in combined.columns if c not in cat_cols + ['id', 'is_train', TARGET]]

# Extract digits from the 10^-4 place up to the 10^3 place
digit_features = []
for c in num_cols:
    scaled = np.rint(combined[c] * 10**4).astype("int64")
    for k in range(-4, 4):
        col_name = f"{c}_digit{k}"
        combined[col_name] = (scaled // 10**(k + 4) % 10).astype('int8')
        # combined[col_name] = (combined[c].fillna(0) // (10**k) % 10).astype('int8')
        digit_features.append(col_name)

# Add the new digit features so they get processed by your frequency/target encoders
num_cols.extend(digit_features)

# Map Original Dataset Target Means
orig_global_mean = orig[TARGET].mean()
for col in cat_cols + num_cols:
    if col in orig.columns:
        real_world_stats = orig.groupby(col, observed=False)[TARGET].mean()
        combined[f"{col}_org_mean"] = combined[col].map(real_world_stats).fillna(orig_global_mean).astype(float)

# Convert Numerics to String Categories
num_to_cat_cols = []
for col in num_cols:
    cat_name = f"{col}_cat"
    combined[cat_name] = combined[col].fillna('NaN').astype(str)
    num_to_cat_cols.append(cat_name)

# Global Frequency Encoding
all_cats = cat_cols + num_to_cat_cols
for col in all_cats:
    freq_mapping = combined[col].value_counts(normalize=True).to_dict()
    combined[f"{col}_fe"] = combined[col].map(freq_mapping).astype(float).fillna(0.0)

# The Mode Collapse Spike
combined['is_30k_spike'] = (combined['Annual_Income_USD'] == 30000.0).astype('int8')

# The Millionaire Cliff (100% buy rate region)
combined['is_millionaire_cliff'] = (combined['Annual_Income_USD'] >= 170537.0).astype('int8')

# The Dead Zone (0% buy rate region)
combined['is_dead_zone'] = ((combined['Annual_Income_USD'] >= 38000.0) & (combined['Annual_Income_USD'] <= 42000.0)).astype('int8')

# Environmental Concern Extremes
combined['is_env_hater'] = (combined['Environmental_Concern_Level'] == 1).astype('int8')

# ==========================================
# Markus's "Smooth Keys" (Binned Numerics)
# ==========================================
print("🔑 Adding Smooth Keys (Income/Commute Bins)...")
combined['income_exact_int'] = np.floor(combined['Annual_Income_USD']).astype(str)
combined['income100_floor']  = np.floor(combined['Annual_Income_USD'] / 100.0).astype(str)
combined['income1000_floor'] = np.floor(combined['Annual_Income_USD'] / 1000.0).astype(str)
combined['commute_integer']  = np.floor(combined['Daily_Commute_km']).astype(str)
# Adding these 4 new string columns to all_cats so they get Frequency and Target Encoded
all_cats.extend(['income_exact_int', 'income100_floor', 'income1000_floor', 'commute_integer'])

train = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test = combined[combined['is_train'] == 0].drop(columns=['is_train', TARGET])

#  FEATURE DROPPING
# Identify numeric columns to evaluate for correlatio (ignore strings/objects because .corr() will fail on them)
eval_cols = [c for c in train.columns if c not in ['id', TARGET] and pd.api.types.is_numeric_dtype(train[c])]

# Find perfectly correlated features (1.0 correlation)
corr_matrix = train[eval_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [column for column in upper_tri.columns if any(upper_tri[column] == 1.0)]

# Find constant features (only 1 unique value) in train or test
to_drop_const = [c for c in train.columns if train[c].nunique() == 1] + \
                [c for c in test.columns if test[c].nunique() == 1]

# Combine all bad features into a set to drop
DROP = set(to_drop_corr).union(set(to_drop_const))
DROP = [c for c in DROP if c not in ['id', TARGET]] # Safety check

if len(DROP) > 0:
    print(f"Dropping {len(DROP)} redundant/constant features")
    # print(f"Dropped features: {{DROP}}")
    train.drop(columns=DROP, inplace=True, errors='ignore')
    test.drop(columns=DROP, inplace=True, errors='ignore')
else:
    print("   -> No redundant features found.")
# ==========================================

FEATURES = [c for c in test.columns if c != 'id']
# Safely remove dropped columns from target encoding list
TARGET_ENCODE_COLS = [c for c in all_cats if c not in DROP] 

print(f"✅ Total Features: {len(FEATURES)}")
print(f"✅ Columns to Target Encode: {len(TARGET_ENCODE_COLS)}")

train

🔑 Adding Smooth Keys (Income/Commute Bins)...
Dropping 104 redundant/constant features
✅ Total Features: 90
✅ Columns to Target Encode: 30


,id,Age,Annual_Income_USD,Daily_Commute_km,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV,Age_digit0,Age_digit1,Annual_Income_USD_digit0,Annual_Income_USD_digit1,Annual_Income_USD_digit2,Annual_Income_USD_digit3,Daily_Commute_km_digit-1,Daily_Commute_km_digit0,Daily_Commute_km_digit1,Charging_Stations_Near_Home_digit0,Charging_Stations_Near_Home_digit1,Charging_Stations_Near_Work_digit0,Charging_Stations_Near_Work_digit1,Gender_org_mean,City_Type_org_mean,Current_Car_Type_org_mean,Home_Charging_Possible_org_mean,Subsidy_Available_org_mean,Range_Anxiety_Level_org_mean,Age_org_mean,Annual_Income_USD_org_mean,Daily_Commute_km_org_mean,Charging_Stations_Near_Home_org_mean,Charging_Stations_Near_Work_org_mean,Environmental_Concern_Level_org_mean,Age_cat,Annual_Income_USD_cat,Daily_Commute_km_cat,Charging_Stations_Near_Home_cat,Charging_Stations_Near_Work_cat,Environmental_Concern_Level_cat,Age_digit0_cat,Age_digit1_cat,Annual_Income_USD_digit0_cat,Annual_Income_USD_digit1_cat,Annual_Income_USD_digit2_cat,Annual_Income_USD_digit3_cat,Daily_Commute_km_digit-1_cat,Daily_Commute_km_digit0_cat,Daily_Commute_km_digit1_cat,Charging_Stations_Near_Home_digit0_cat,Charging_Stations_Near_Home_digit1_cat,Charging_Stations_Near_Work_digit0_cat,Charging_Stations_Near_Work_digit1_cat,Environmental_Concern_Level_digit0_cat,Gender_fe,City_Type_fe,Current_Car_Type_fe,Home_Charging_Possible_fe,Subsidy_Available_fe,Range_Anxiety_Level_fe,Age_cat_fe,Annual_Income_USD_cat_fe,Daily_Commute_km_cat_fe,Charging_Stations_Near_Home_cat_fe,Charging_Stations_Near_Work_cat_fe,Environmental_Concern_Level_cat_fe,Age_digit0_cat_fe,Age_digit1_cat_fe,Annual_Income_USD_digit0_cat_fe,Annual_Income_USD_digit1_cat_fe,Annual_Income_USD_digit2_cat_fe,Annual_Income_USD_digit3_cat_fe,Daily_Commute_km_digit-1_cat_fe,Daily_Commute_km_digit0_cat_fe,Daily_Commute_km_digit1_cat_fe,Charging_Stations_Near_Home_digit0_cat_fe,Charging_Stations_Near_Home_digit1_cat_fe,Charging_Stations_Near_Work_digit0_cat_fe,Charging_Stations_Near_Work_digit1_cat_fe,is_30k_spike,is_millionaire_cliff,is_dead_zone,is_env_hater,income_exact_int,income100_floor,income1000_floor,commute_integer
0,0,66,92887.0,23.4,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,0.0,6,6,7,8,8,2,4,3,2,3,0,7,0,0.170929,0.176817,0.172250,0.200714,0.025301,0.198676,0.200913,0.000000,0.181818,0.154994,0.166913,0.021361,66,92887.0,23.4,3,7,1.0,6,6,7,8,8,2,4,3,2,3,0,7,0,1,0.550232,0.382160,0.453965,0.69214,0.371583,0.903671,0.022189,0.000156,0.001025,0.088105,0.067977,0.220470,0.119046,0.218299,0.089344,0.098165,0.136997,0.114493,0.089187,0.068507,0.155343,0.119869,0.839831,0.095176,0.742066,0,0,0,1,92887.0,928.0,92.0,23.0
1,1,38,30000.0,5.0,2,2,4.0,Male,Rural,SUV,Yes,No,Low,0.0,8,3,0,0,0,0,0,5,0,2,0,2,0,0.170929,0.189840,0.181010,0.200714,0.025301,0.198676,0.175355,0.066427,0.191517,0.187404,0.188034,0.252891,38,30000.0,5.0,2,2,4.0,8,3,0,0,0,0,0,5,0,2,0,2,0,4,0.550232,0.185583,0.368738,0.69214,0.371583,0.903671,0.020044,0.091999,0.215599,0.151632,0.074310,0.195549,0.107725,0.218350,0.199227,0.149168,0.146475,0.158213,0.292327,0.290030,0.218171,0.183997,0.839831,0.100067,0.742066,1,0,0,0,30000.0,300.0,30.0,5.0
2,2,26,94389.0,36.8,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,1.0,6,2,9,8,3,4,8,6,3,8,0,5,1,0.180302,0.169197,0.172250,0.133698,0.274467,0.198676,0.200000,1.000000,0.285714,0.195710,0.166667,0.414046,26,94389.0,36.8,8,15,5.0,6,2,9,8,3,4,8,6,3,8,0,5,1,5,0.441876,0.432257,0.453965,0.30786,0.628417,0.903671,0.022353,0.000063,0.002892,0.031121,0.024436,0.191398,0.119046,0.108970,0.090048,0.098165,0.073144,0.109598,0.076601,0.083534,0.184213,0.031121,0.839831,0.094157,0.257934,0,0,0,0,94389.0,943.0,94.0,36.0
3,3,66,73580.0,23.7,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,0.0,6,6,0,8,5,3,7,3,2,6,0,9,0,0.170929,0.176817,0.174655,0.200714,0.025301,0.198676,0.200913,1.000000,0.300000,0.159174,0.178470,0.

# 3. 10 FOLD CV WITH SKLEARN TARGET ENCODING

In [4]:
%%time

MODEL_NAME = 'LGBMgoss'
FOLDS = 10
print(f"\n🚀 Training {MODEL_NAME} with {FOLDS } Folds...")

X = train[FEATURES]
y = train[TARGET]
X_test = test[FEATURES]

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]
    X_test_fold = X_test.copy()

    
    # Triple Sklearn Target Encoders (Auto, Strict 10, and Massive 100)
    te_auto = TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=42)
    te_10   = TargetEncoder(shuffle=True, cv=5, smooth=10.0, random_state=42)
    te_100  = TargetEncoder(shuffle=True, cv=5, smooth=100.0, random_state=42)

    X_train_enc_auto = te_auto.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc_auto = te_auto.transform(X_valid[TARGET_ENCODE_COLS])
    X_test_enc_auto  = te_auto.transform(X_test_fold[TARGET_ENCODE_COLS])

    X_train_enc_10 = te_10.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc_10 = te_10.transform(X_valid[TARGET_ENCODE_COLS])
    X_test_enc_10  = te_10.transform(X_test_fold[TARGET_ENCODE_COLS])

    X_train_enc_100 = te_100.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc_100 = te_100.transform(X_valid[TARGET_ENCODE_COLS])
    X_test_enc_100  = te_100.transform(X_test_fold[TARGET_ENCODE_COLS])
    
    for i, col in enumerate(TARGET_ENCODE_COLS):
        # Auto smoothing TE
        X_train[f"{col}_TE_auto"] = X_train_enc_auto[:, i].astype('float32')
        X_valid[f"{col}_TE_auto"] = X_valid_enc_auto[:, i].astype('float32')
        X_test_fold[f"{col}_TE_auto"] = X_test_enc_auto[:, i].astype('float32')
        
        # Strict (10.0) smoothing TE
        X_train[f"{col}_TE_10"] = X_train_enc_10[:, i].astype('float32')
        X_valid[f"{col}_TE_10"] = X_valid_enc_10[:, i].astype('float32')
        X_test_fold[f"{col}_TE_10"] = X_test_enc_10[:, i].astype('float32')

        # Massive (100.0) smoothing TE (Markus's method)
        X_train[f"{col}_TE_100"] = X_train_enc_100[:, i].astype('float32')
        X_valid[f"{col}_TE_100"] = X_valid_enc_100[:, i].astype('float32')
        X_test_fold[f"{col}_TE_100"] = X_test_enc_100[:, i].astype('float32')
        
        # Drop the original string column
        X_train.drop(columns=[col], inplace=True)
        X_valid.drop(columns=[col], inplace=True)
        X_test_fold.drop(columns=[col], inplace=True)

    # --- LIGHTGBM MODEL ---
    bag_params = {
        'data_sample_strategy': "bagging",
        'n_estimators': 20000,
        'learning_rate': 0.02,
        'max_depth': 5,
        'num_leaves': 32, 
        'min_child_samples': 10,
        'subsample': 0.812763,
        'colsample_bytree': 0.30293,
        'reg_alpha': 0.07094,
        'reg_lambda': 2.03303,
        'max_bin': 1024,
        'random_state': 42,
        'feature_pre_filter': False,
        'metric': 'auc',
        'n_jobs': -1,
        'verbose': -1
    }

    goss_params = {
        'data_sample_strategy': "goss",
        'top_rate': 0.6,
        'other_rate': 0.15,
        'n_estimators': 20000,
        'learning_rate': 0.02,
        'max_depth': -1, #
        'num_leaves': 127, #
        'min_child_samples': 40, #
        'subsample': 0.812763,
        'colsample_bytree': 0.30293,
        'reg_alpha': 0.07094,
        'reg_lambda': 2.03303,
        'max_bin': 1024,
        'random_state': 42,
        'feature_pre_filter': False,
        'metric': 'auc',
        'n_jobs': -1,
        'verbose': -1
    }
    
    clf = lgb.LGBMClassifier(**bag_params if 'bag' in MODEL_NAME else goss_params)        
    
    clf.fit(
        X_train, y_train, 
        eval_set=[(X_valid, y_valid)], 
        callbacks=[
            lgb.early_stopping(stopping_rounds=500, verbose=False),
            lgb.log_evaluation(period=1000)
        ]
    )

    oof_preds[valid_idx] = clf.predict_proba(X_valid)[:, 1]
    test_preds += clf.predict_proba(X_test_fold)[:, 1] / FOLDS 
    
    fold_auc = roc_auc_score(y_valid, oof_preds[valid_idx])
    print(f"   --> Fold {fold} CONVERGED at Tree #{clf.best_iteration_} | AUC: {fold_auc:.6f}")

oof_auc = roc_auc_score(y, oof_preds)

print("\n" + "="*45)
print(f"🏆   {MODEL_NAME} OOF AUC: {oof_auc:.6f}")
print("="*45)


🚀 Training LGBMgoss with 10 Folds...
   --> Fold 1 CONVERGED at Tree #395 | AUC: 0.944895
[1000]	valid_0's auc: 0.945202
   --> Fold 2 CONVERGED at Tree #520 | AUC: 0.945269
[1000]	valid_0's auc: 0.945373
   --> Fold 3 CONVERGED at Tree #550 | AUC: 0.945467
[1000]	valid_0's auc: 0.945936
   --> Fold 4 CONVERGED at Tree #530 | AUC: 0.945972
   --> Fold 5 CONVERGED at Tree #453 | AUC: 0.946570
[1000]	valid_0's auc: 0.947061
   --> Fold 6 CONVERGED at Tree #562 | AUC: 0.947088
[1000]	valid_0's auc: 0.945814
   --> Fold 7 CONVERGED at Tree #508 | AUC: 0.945950
   --> Fold 8 CONVERGED at Tree #457 | AUC: 0.946982
   --> Fold 9 CONVERGED at Tree #473 | AUC: 0.947008
[1000]	valid_0's auc: 0.945039
   --> Fold 10 CONVERGED at Tree #530 | AUC: 0.945135

🏆   LGBMgoss OOF AUC: 0.946024
CPU times: user 1h 24min 54s, sys: 49.2 s, total: 1h 25min 43s
Wall time: 29min 22s


# 4. SAVE SUBMISSION AND OOF PREDICTIONS

In [5]:
print("\n" + "="*45)
print(f"🏆   {MODEL_NAME} OOF AUC: {oof_auc:.6f}")
print("="*45)

MODEL_NAME = f"{MODEL_NAME}_Triple_TE"

# Save Kaggle Submission (Averaged Test Predictions)
submission[TARGET] = test_preds
submission.to_csv(f"submission_{MODEL_NAME}_{oof_auc:.6f}.csv", index=False)
print(f"💾 Saved 'submission_{MODEL_NAME}_{oof_auc:.6f}.csv'")

# Save OOF Predictions (for ensembling meta-model)
# oof_df = pd.DataFrame({'id': train['id'], 'OOF_Pred': oof_preds})
# oof_df.to_csv(f'oof_{MODEL_NAME}.csv', index=False)
np.save(f"oof_{MODEL_NAME}_{oof_auc:.6f}.npy", oof_preds)
print(f"💾 Saved 'oof_{MODEL_NAME}_{oof_auc:.6f}.npy'")

# Save Raw Test Predictions (for ensembling)
# test_df = pd.DataFrame({'id': test['id'], TARGET: test_preds})
# test_df.to_csv(f'test_{MODEL_NAME}.csv', index=False)
np.save(f"test_{MODEL_NAME}_{oof_auc:.6f}.npy", test_preds)
print(f"💾 Saved 'test_{MODEL_NAME}_{oof_auc:.6f}.npy'")


🏆   LGBMgoss OOF AUC: 0.946024
💾 Saved 'submission_LGBMgoss_Triple_TE_0.946024.csv'
💾 Saved 'oof_LGBMgoss_Triple_TE_0.946024.npy'
💾 Saved 'test_LGBMgoss_Triple_TE_0.946024.npy'
